In [ ]:
import yfinance as yf 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

yfinance fetches stock data from Yahoo Finance as
Numpy is for tha mathematics 
Pandas for dataframes (create arrays later)
matplotlib for my charts

In [ ]:
years = 15 
endDate = dt.datetime.now()
startDate = endDate - dt. timedelta(days = 365 * years) 

15 years worth of data is to be used (to account for volatility and other uncertainties)
[the time delta logic]

In [ ]:
tickers = ['BBCA.JK', 'CDIA.JK', 'BUMI.JK', 'BRPT.JK', 'PTRO.JK']

adj_close_df = pd.DataFrame()

for ticker in tickers:
    data = yf.download(ticker, start = startDate , end = endDate, auto_adjust = True)
    adj_close_df[ticker] = data['Close']

print(data.head())
print(adj_close_df.head())

adj_close_df is our dataframe, creates rows and columns 
auto_adjust is used for the closing price to take account for splits and dividents

In [ ]:
log_returns = np. log(adj_close_df / adj_close_df.shift(1))
log_returns = log_returns.dropna()

print(log_returns.head()) # difference bewtween today and tommorrow

shift(1) is used to compare the value of the current row with the one before
we use logarithms to compare, due to the nature of returns which should be time-additive and normally distributed, which will help later on when the monte-carlo sumulation happens

In [ ]:
portofolio_value = 1000000000
weights = np.array([1/len(tickers)]*len(tickers))
print(weights)

main formula: log(P_t / P_{t-1}) | this is just to make sure each portfolio has equal weigth from that Rp1Billion.
Usually real portfolios get optimized 

In [ ]:
cov_matrix = log_returns.cov()
print(cov_matrix)

this creates the covariance matrix, output will be a table where the top and most left of the table are tickers -> and the numbers inside represent their correlation:
positive correlation, means they move together
negative correlation, means they move opposite of each other (good for diversification)
the bigger the number, the stronger the correlation is

In [ ]:
def expected_return(weights, log_returns):
    return np.sum(log_returns.mean()*weights) 


portofolio_expected_return = expected_return(weights, log_returns)
print(portofolio_expected_return)

In [ ]:
def standard_deviation(weights, cov_matrix):
    variance = weights.T @ cov_matrix @weights
    return np.sqrt(variance)
portofolio_std_dev = standard_deviation(weights, cov_matrix)
print(portofolio_std_dev)

portofolio standard deviation formula: σ_p = sqrt(wᵀ Σ w) | "w" is the weights vector, and 
"Σ" is the covariance matrix. From Modern Portfolio Theory

In [ ]:
def random_z_score():
    return np.random.normal(0,1) # take any number from standard bell curve

the Z-scores are random numbers we get from the normal distribution table

In [ ]:
days = 5

def scenario_gain_loss(portofolio_value, portofolio_expected_return, z_score ,portofolio_std_dev,days):
    return (portofolio_value * portofolio_expected_return) + (portofolio_value * portofolio_std_dev * z_score * np.sqrt(days))

main formula for the gain/loss: P * μ + P * σ * z * sqrt(days)
square root the (days) is to scale daily volatility, because no way if gain 5% for the 5 days, the gain were constant at 5% (not realistic)

In [ ]:
simulations = 10000
scenarioReturn =[]
for i in range(simulations):
    z_score = random_z_score()
    scenarioReturn.append(scenario_gain_loss(portofolio_value, portofolio_expected_return, z_score ,portofolio_std_dev, days))

In [ ]:
confidence_interval = 0.95
VaR = -np.percentile(scenarioReturn , 100*(1- confidence_interval))
print(VaR)

VaR is our solution, the 95% means: there is a 5% chance the portfolio loses more than Rp X over 5 days. 
the limitation is that it says does not took account for more bad losses that could potential happen beyond that treshold can get 

In [ ]:
plt.hist(scenarioReturn, bins=50, density = True )
plt.xlabel('Scenario Gain/Loss(IDR)')
plt.ylabel('Frequency')
plt.title(f'Distribution of Portfolio Gain/Loss over {days} Days')
plt.axvline(-VaR, color= 'r', linestyle = 'dashed', linewidth =2, label=f'VaR at {confidence_interval:.0%} confidence level')
plt.legend()
plt.show()

In [ ]:
var_idr = np.percentile(scenarioReturn, 100 * (1 - confidence_interval))
scenarioReturn_array = np.array(scenarioReturn)

exceptions = scenarioReturn_array[scenarioReturn_array < var_idr]
es_idr = np.mean(exceptions)

var_percent = var_idr / portofolio_value
es_percent = es_idr / portofolio_value

print("--- 5-day Risk Results (IDR)---")
print(f"95% VaR: Rp {abs(var_idr):,.0f}")
print(f"95% Expected Shortfall: Rp {abs(es_idr):,.0f}")
print(f"--- Percentage of Portfolio ---")
print(f"VaR %: {abs(var_percent):.2%}")
print(f"ES %: {abs(es_percent):.2%}")

this is where the limitation that I said before will be resolved by finding the "Expected Shortfal" also called CVaR
-> it is the average loss of the WORST 5% of scenarios. and now ES is used more and more than normal VaR


In [ ]:
historical_portfolio_returns = log_returns.dot(weights)
daily_VaR_pct = np.percentile(historical_portfolio_returns, 5)
historical_exceptions = historical_portfolio_returns[historical_portfolio_returns < daily_VaR_pct]
num_exceptions = len(historical_exceptions)
exception_rate = num_exceptions / len(historical_portfolio_returns)

print(f"\n--- Historical Backtest Results (1-Day) ---")
print(f"Total Days Analyzed: {len(historical_portfolio_returns)}")
print(f"Historical Exceptions: {num_exceptions}")
print(f"Actual Exception Rate: {exception_rate:.2%}") 
#If the rate is near 5%, the model is well-calibrated

this is the back testing algorithm. 
checking historical returns and counting how many days actually nreached the VaR treshold. if the exception rate is close to 5% -> means this model is well-calibrated

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(historical_portfolio_returns, label='Daily Portfolio Returns', alpha=0.7)
plt.axhline(daily_VaR_pct, color='r', linestyle='--', label=f'95% VaR Line ({daily_VaR_pct:.2%})')

# Mark the exceptions as red dots
exceptions_dates = historical_portfolio_returns[historical_portfolio_returns < daily_VaR_pct]
plt.scatter(exceptions_dates.index, exceptions_dates.values, color='red', label='Exceptions (Breaches)')

plt.title(f'Historical Backtest: {num_exceptions} Exceptions found ({exception_rate:.2%})')
plt.legend()
plt.show()